# Viking 2 reference values (PDS Atmospheres Node)
Computes the Mars reference values (air temperature, wind, pressure, optical depth) used in the Earth-analog comparison.

Upload from PDS volume `vl_1001`, folder `data`: `vl_tbin.dat/.lbl`, `vl_avep.dat/.lbl`, `vl_opac.dat/.lbl`. Then run the cells in order.

Datasets: `VL1/VL2-M-MET-4-BINNED-P-T-V-V1.0`, `VL1/VL2-M-MET-4-DAILY-AVG-PRESSURE-V1.0`, `VL1/VL2-M-LCS-5-ATMOS-OPTICAL-DEPTH-V1.0`.

## 1) Upload the six files (select all at once)

In [ ]:
from google.colab import files
import os
uploaded = files.upload()
print(sorted(os.listdir(".")))

## 2) Clean file names (the browser can add extra text to the names)

In [ ]:
import os, shutil
for base in ["vl_tbin", "vl_avep", "vl_opac"]:
    for ext in [".dat", ".lbl"]:
        c = [f for f in os.listdir(".") if f.lower().startswith(base) and f.lower().endswith(ext) and f != base + ext]
        if c:
            shutil.copy(max(c, key=os.path.getmtime), base + ext)
        print(base + ext, "OK" if os.path.exists(base + ext) else "MISSING", os.path.getsize(base + ext) if os.path.exists(base + ext) else "")

## 3) Extract the Mars reference values (Viking 2)
Only quality-flagged-good bins are used (missing, interpolated and lander-interference bins are excluded).
Averages are balanced by season (12 bins of solar longitude) and by time of day, so seasons sampled more often do not bias the result.

In [ ]:
import re, numpy as np, pandas as pd

def parse_cols2(lbl_path):
    cols, cur = [], None
    for line in open(lbl_path, "r", errors="replace"):
        s = line.strip()
        if re.match(r"OBJECT\s*=\s*COLUMN\b", s, re.I):
            cur = {}
        elif cur is not None and re.match(r"END_OBJECT", s, re.I):
            if "name" in cur and "start" in cur and "bytes" in cur:
                cols.append(cur)
            cur = None
        elif cur is not None:
            m = re.match(r"(NAME|START_BYTE|BYTES|ITEMS)\s*=\s*(.+)$", s, re.I)
            if m:
                k, val = m.group(1).upper(), m.group(2).strip().strip('"')
                if k == "NAME": cur["name"] = val
                elif k == "START_BYTE": cur["start"] = int(val)
                elif k == "BYTES": cur["bytes"] = int(val)
                else: cur["items"] = val
    return cols

def read_table2(dat, lbl):
    cols = [c for c in parse_cols2(lbl) if "items" not in c]
    specs = [(c["start"] - 1, c["start"] - 1 + c["bytes"]) for c in cols]
    df = pd.read_fwf(dat, colspecs=specs, names=[c["name"] for c in cols], header=None)
    for c in df.columns:
        conv = pd.to_numeric(df[c], errors="coerce")
        if conv.notna().sum() >= 0.9 * df[c].notna().sum():
            df[c] = conv
    return df

def bal_mean(df, col, hr_col=None, nbins=12):
    d = df.copy()
    d["_ls"] = ((d["SOL_LON"] % 360) // (360 / nbins)).astype(int)
    if hr_col:
        per = d.groupby(["_ls", hr_col])[col].mean().groupby(level=0).mean()
    else:
        per = d.groupby("_ls")[col].mean()
    return per.mean()

print("columns found: tbin", len(parse_cols2("vl_tbin.lbl")), "| avep", len(parse_cols2("vl_avep.lbl")), "| opac", len(parse_cols2("vl_opac.lbl")))

tbin = read_table2("vl_tbin.dat", "vl_tbin.lbl")
v = tbin[tbin["SC_ID"] == "VL2"]
t = v[v["T_NLI_FLAG"].isna() & (v["TAMB_NLI_MN"] > 100)]
t_mean = bal_mean(t, "TAMB_NLI_MN", hr_col="LOCAL_HR_BIN")
sea = t.assign(_ls=((t["SOL_LON"] % 360) // 30).astype(int)).groupby(["_ls", "LOCAL_HR_BIN"])["TAMB_NLI_MN"].mean()
diurnal = (sea.groupby(level=0).max() - sea.groupby(level=0).min()).mean()
w = v[v["WIND_FLG"].isna() & (v["WND_TMP_NLI"] > 0) & (v["WND_SPD_MEAN"] >= 0)]
w_mean = bal_mean(w, "WND_SPD_MEAN", hr_col="LOCAL_HR_BIN")

avep = read_table2("vl_avep.dat", "vl_avep.lbl")
p = avep[(avep["SC_ID"] == "VL2") & (avep["PRESS_MEAN"] > 0)]
p_mean = bal_mean(p, "PRESS_MEAN")

opac = read_table2("vl_opac.dat", "vl_opac.lbl")
o = opac[(opac["SC_ID"] == "VL2") & (opac["OPACITY"] > 0)].copy()
o["_ls"] = ((o["SOL_LON"] % 360) // 30).astype(int)
o_med = o.groupby("_ls")["OPACITY"].median().mean()
lsv = o["SOL_LON"] % 360

print("good temperature bins:", len(t), "| good wind bins:", len(w), "| pressure days:", len(p), "| opacity points:", len(o))
print(f"Air temperature (1.6 m), balanced mean: {t_mean:.1f} K ({t_mean-273.15:.1f} C) | min/max {t['TAMB_NLI_MN'].min():.1f}/{t['TAMB_NLI_MN'].max():.1f} K | day-night range ~{diurnal:.1f} K")
print(f"Wind speed, balanced mean: {w_mean:.2f} m/s | 95th pct: {w['WND_SPD_MEAN'].quantile(.95):.1f} m/s")
print(f"Pressure, balanced mean: {p_mean:.2f} mbar = {p_mean*100:.0f} Pa | daily-mean range {p['PRESS_MEAN'].min():.2f} to {p['PRESS_MEAN'].max():.2f} mbar")
print(f"Optical depth, balanced median: {o_med:.2f} | Ls 0-180: {o[lsv<180]['OPACITY'].median():.2f} | Ls 180-360: {o[lsv>=180]['OPACITY'].median():.2f} | max {o['OPACITY'].max():.1f}")
print("\nMARS = {")
print(f'    "ts_mean_K": {t_mean:.1f},')
print('    "elevation_m": -4000,')
print(f'    "dust_aod": {o_med:.2f},')
print('    "lai": 0.0,')
print('    "soil_wetness": 0.0,')
print(f'    "wind_ms": {w_mean:.2f},')
print(f'    "pressure_Pa": {p_mean*100:.0f},')
print("}")